# ComfyUI AI Influencer on Kaggle

This notebook sets up ComfyUI for AI influencer generation on Kaggle with T4 x2 GPUs.

### Instructions:
1. Enable T4 x2 GPU.
2. Turn on Internet Access.
3. Run cells sequentially.

In [ ]:
!git clone https://github.com/Dhruwkd/comfyui-kaggle-influencer.git
%cd comfyui-kaggle-influencer
!chmod +x scripts/*.sh
!./scripts/install_nodes.sh
!./scripts/download_models.sh

In [ ]:
# Install cloudflared binary to a local path (avoid "Text file busy" on /usr/local/bin)
!mkdir -p /kaggle/working/bin
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /kaggle/working/bin/cloudflared
!chmod +x /kaggle/working/bin/cloudflared

import subprocess
import threading
import time
import os
import sys
import re

# Verify ComfyUI exists
comfyui_path = "/kaggle/working/ComfyUI"
if not os.path.exists(comfyui_path):
    raise RuntimeError(f"ComfyUI not found at {comfyui_path}. Did cell 1 complete successfully?")

tunnel_url = {"value": None}
tunnel_ready = threading.Event()

def run_tunnel():
    proc = subprocess.Popen(
        ["/kaggle/working/bin/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://localhost:8188"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    for line in proc.stdout:
        line = line.strip()
        print(f"[cloudflared] {line}", flush=True)
        if "trycloudflare.com" in line and not tunnel_url["value"]:
            match = re.search(r'(https://[a-zA-Z0-9\-]+\.trycloudflare\.com)', line)
            if match:
                tunnel_url["value"] = match.group(1)
                tunnel_ready.set()

tunnel_thread = threading.Thread(target=run_tunnel, daemon=True)
tunnel_thread.start()

# Wait for tunnel URL (max 30 seconds)
print("⏳ Waiting for cloudflared tunnel to establish...", flush=True)
tunnel_ready.wait(timeout=30)

if tunnel_url["value"]:
    url = tunnel_url['value']
    print(f"\n{'#'*70}", flush=True)
    print(f"#  🌐  COMFYUI TUNNEL URL — COPY THIS LINK:", flush=True)
    print(f"#", flush=True)
    print(f"#      {url}", flush=True)
    print(f"#", flush=True)
    print(f"#  👆  OPEN THIS IN YOUR BROWSER (or Claude's Browser pane)", flush=True)
    print(f"#  ❌  DO NOT USE: http://localhost:8188  OR  http://0.0.0.0:8188", flush=True)
    print(f"#  ❌  THOSE ARE INTERNAL ADDRESSES — THEY WILL NOT WORK", flush=True)
    print(f"{'#'*70}\n", flush=True)
else:
    print("⚠️ Tunnel URL not detected. Check [cloudflared] logs above.", flush=True)

# Give ComfyUI a moment to be ready
time.sleep(3)

# Start ComfyUI
os.chdir(comfyui_path)
print(f"🚀 Starting ComfyUI on port 8188 (bound to 0.0.0.0 for tunnel)...", flush=True)
if tunnel_url["value"]:
    print(f"📱 YOUR COMFYUI ADDRESS: {tunnel_url['value']}", flush=True)
!python main.py --lowvram --port 8188 --listen 0.0.0.0